In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
-- Crear widget para datalake
create widget text storageName default "adlsfinalproject";

In [0]:
# Obtener widget
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
--Crear External location para el Data Lake. 
--Obs: Databricks ya creó un External Location para el metastore, por tanto debemos cambiar al nombre "catalog". 
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-catalog`
URL 'abfss://catalog@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
-- Crear ext. location para raw
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
-- Crear ext. location para bronze
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
-- Crear ext. location para silver
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
-- Crear ext. location para golden
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
-- Borrar catalog si existe
DROP CATALOG IF EXISTS catalog_cr CASCADE;

In [0]:
%sql
-- Crear catalog. 
-- Se cambia el nombre del managed location de "metastore" por "catalog"
CREATE CATALOG IF NOT EXISTS catalog_cr
MANAGED LOCATION 'abfss://catalog@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
-- Borrar schemas en caso de existir.
DROP SCHEMA IF EXISTS catalog_cr.raw;
DROP SCHEMA IF EXISTS catalog_cr.bronze;
DROP SCHEMA IF EXISTS catalog_cr.silver;
DROP SCHEMA IF EXISTS catalog_cr.golden;

In [0]:
# Borra archivos en los containers bronze, silver y gold.
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
-- Crear los schemas del catalog
CREATE SCHEMA IF NOT EXISTS catalog_cr.raw;
CREATE SCHEMA IF NOT EXISTS catalog_cr.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_cr.silver;
CREATE SCHEMA IF NOT EXISTS catalog_cr.golden;

###Tablas Bronze

In [0]:
%sql
-- Crear tabla crime_reports
CREATE TABLE IF NOT EXISTS catalog_cr.bronze.crime_reports (
report_id integer,
report_year integer,
month_id integer,
department string,
province string,
district string,
ubigeo string,
crime_id integer,
n_crimes integer,
emergency_district boolean,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/crime_reports"

In [0]:
%sql
-- Crear tabla crimes
CREATE TABLE IF NOT EXISTS catalog_cr.bronze.crimes (
crime_id integer,
crime string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/crimes"

In [0]:
%sql
-- Crear tabla month_report
CREATE TABLE IF NOT EXISTS catalog_cr.bronze.months (
  month_id integer,
  month_name string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/months"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_cr.silver.crime_reports_transformed (
  report_id integer,
  ubigeo string,
  department string,
  province string,
  district string,
  crime_name string,
  report_year integer,
  total_n_crimes integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/crime_reports_transformed"


###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_cr.golden.extortion_reports (
  report_id integer,
  ubigeo string,
  department string,
  province string,
  district string,
  total_extortions integer,
  min_extortions integer,
  max_extortions integer,
  year_min_extortions integer,
  year_max_extortions integer,
  avg_extortions double
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/extortion_reports"